# Notebook 04 — Model Benchmarking: Manual + LazyPredict + FLAML + PyCaret

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project root:', project_root)


## Model Benchmarking

### Definition
Model Benchmarking in this project means we design, validate, serve, and observe ML predictions through stable HTTP interfaces.

### Theory
APIs decouple producers (model service) and consumers (web app, batch jobs, partner systems). Typed contracts reduce ambiguity and production failures.

### Motivation
A single model baseline can hide significant quality/performance gains from alternatives.

### Real-World Example
Production teams compare multiple algorithms before locking serving contract and SLO budgets.

### Visual Explanation
Diagram/code cell below shows component flow and responsibilities.

### Code Explanation
Code cells in this section are structured as: setup → implementation → validation output.

### Best Practices
- Keep contracts explicit and versioned.
- Validate early at boundary.
- Log request IDs for traceability.
- Measure latency, error rate, and throughput.

### Common Mistakes
- Benchmarking on test split repeatedly.
- Ignoring runtime and deployment complexity in model selection.


In [ ]:
from src.data import load_california_housing, split_dataset
from src.training import train_and_rank_models
from src.benchmarking import run_lazypredict_benchmark, run_flaml_benchmark, run_pycaret_benchmark

X, y = load_california_housing()
split = split_dataset(X, y, random_state=42)
trained, ranking = train_and_rank_models(split.X_train, split.y_train, split.X_val, split.y_val)
display(ranking.head(10))

In [ ]:
lazy = run_lazypredict_benchmark(split.X_train, split.X_val, split.y_train, split.y_val)
flaml = run_flaml_benchmark(split.X_train, split.X_val, split.y_train, split.y_val, time_budget_seconds=30)
pyc = run_pycaret_benchmark(split.X_train, split.X_val, split.y_train, split.y_val)

print('LazyPredict:', lazy.status, lazy.notes)
print('FLAML:', flaml.status, flaml.notes)
print('PyCaret:', pyc.status, pyc.notes)
if not lazy.table.empty:
    display(lazy.table.head())
if not flaml.table.empty:
    display(flaml.table.head())
if not pyc.table.empty:
    display(pyc.table.head())

## Tradeoff Discussion

- LazyPredict: fastest broad scan, low control.
- FLAML: efficient AutoML with time budget control.
- PyCaret: high-level experimentation workflow, heavier dependency/runtime footprint.